# 04 · Take-home project: build your own trading dashboard

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kolyamkl/qm-fintech-quant-workshop/blob/main/notebooks/04_project_trading_dashboard.ipynb)

*QM FinTech Society quant workshop · a project for your free time*

You'll build a **trading dashboard**: a web app that shows how an investment performed, how risky it was, and how much of its movement is just the market. A one-page report like this is often called a **tearsheet**.

**Two parts:**
1. **This notebook:** write each calculation as a small Python function. Every step has a 💡 hint and a check that tells you ✅ or ❌.
2. **The web app:** put your functions into a Streamlit app and run it on your own computer from a terminal.

You can do Part 1 in Colab or on your laptop (`jupyter lab`); Part 2 needs your laptop.

**How each step works:** a ✏️ cell holds a function with `...` where your code goes. Replace the `...`, run the cell, and read the check underneath your code. Stuck? Every answer is in [`solution/analytics.py`](https://github.com/kolyamkl/qm-fintech-quant-workshop/blob/main/projects/trading_dashboard/solution/analytics.py).

> **Running in Google Colab?** If Colab warns that this notebook wasn't written by Google, click **Run anyway**. Then use **File → Save a copy in Drive** so your changes are kept.

## Setup

This first cell gets the notebook ready. In Google Colab it downloads the workshop code and price data from GitHub; on a laptop it simply finds them. It also imports pandas (tables of data) and matplotlib (charts), which the rest of the notebook uses.

In [ ]:
# ▶ Setup — run this cell first.
# In Google Colab it downloads the workshop repo; on your laptop it just finds the code.
import os, subprocess, sys

REPO = "qm-fintech-quant-workshop"
if "google.colab" in sys.modules and os.path.basename(os.getcwd()) != REPO:
    if not os.path.exists(REPO):
        subprocess.run(["git", "clone", "-q", f"https://github.com/kolyamkl/{REPO}.git"], check=True)
    os.chdir(REPO)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")  # the code lives one folder up from notebooks/
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

# Re-load your edited .py files automatically (handy when you change strategies/template.py).
# Skipped where it can't run (Colab's IPython version doesn't support it on Python 3.13).
try:
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")
except ImportError:
    pass

import matplotlib.pyplot as plt
import pandas as pd
pd.set_option("display.precision", 4)

## Step 0. Pick your data

A dashboard needs something to analyse and something to compare it with. This cell loads Apple's (AAPL) closing prices as the investment and SPY (the S&P 500) as the benchmark. It also imports numpy (for square roots) and `check`, which tests your functions.

In [ ]:
import numpy as np
from backtester import load_prices
from projects.trading_dashboard.checks import check

TICKER = "AAPL"  # ✏️ later, try any ticker from notebook 01
prices = load_prices(TICKER)["Close"]
benchmark = load_prices("SPY")["Close"]
prices.tail()

## Step 1. Returns

Every number on the dashboard starts from daily **returns**: the % change from one close to the next. Write `to_returns` so it gives those daily changes. The first day has no day before it, so its return is empty: drop it.

💡 **Hint:** pandas has a method for % change, and `.dropna()` removes empty values.

In [ ]:
def to_returns(prices):
    """Daily % changes of a price series, without the empty first day."""
    returns = ...  # ✏️ your code here
    return returns


check(to_returns)

## Step 2. Equity curve

If you had invested $1 at the start, what would it be worth each day? Returns **compound**: each day's value is the day before's times (1 + today's return). Write `equity_curve` to give that value for every day.

💡 **Hint:** `1 + returns`, then `.cumprod()` multiplies the values together day by day.

In [ ]:
def equity_curve(returns):
    """Growth of $1: compound the daily returns."""
    equity = ...  # ✏️
    return equity


check(equity_curve)

## Step 3. Drawdown

**Drawdown** is how far the equity is below its highest value so far: 0 on a new high, −0.2 when it's 20% below. Its lowest value is the **max drawdown**, the most painful loss you would have sat through.

💡 **Hint:** `.cummax()` gives the highest value so far on each day.

In [ ]:
def drawdown(equity):
    """How far below its highest value so far the equity is each day (0 = at a new high)."""
    dd = ...  # ✏️
    return dd


check(drawdown)

## Step 4. Annual return and volatility

**Annual return (CAGR):** multiply all the (1 + return) values together, raise that to the power 252 / number of days, and subtract 1. 252 is the number of trading days in a year. **Annual volatility:** the standard deviation of the daily returns times √252.

💡 **Hint:** `.prod()`, `len(returns)`, `**` for powers, `.std()` and `np.sqrt(252)`.

In [ ]:
def annual_return(returns):
    """CAGR: the steady yearly return that gives the same end result."""
    return ...  # ✏️


def annual_volatility(returns):
    """How much daily returns swing, scaled to a year."""
    return ...  # ✏️


check(annual_return)
check(annual_volatility)

## Step 5. Sharpe, Sortino and hit rate

**Sharpe ratio:** the average daily return divided by the standard deviation of daily returns, times √252 (we take the risk-free rate as 0, as in notebook 01). **Sortino ratio:** the same, but using the standard deviation of the *down* days only, because investors mind losses more than gains. **Hit rate:** the share of days that went up.

💡 **Hint:** `returns[returns < 0]` keeps only the down days. `(returns > 0)` is True or False per day, and the `.mean()` of that is the share of True.

In [ ]:
def sharpe(returns):
    """Average daily return per unit of daily volatility, scaled to a year."""
    return ...  # ✏️


def sortino(returns):
    """Like Sharpe, but only the down days count as risk."""
    return ...  # ✏️


def hit_rate(returns):
    """Share of days with a positive return."""
    return ...  # ✏️


check(sharpe)
check(sortino)
check(hit_rate)

## Step 6. Rolling numbers

One Sharpe ratio for ten years hides the good and bad stretches. A **rolling** version recalculates it every day using only the last `window` days, so you can draw it as a line. The dashboard uses 63 days (about three months) for Sharpe and 30 days for volatility.

💡 **Hint:** `returns.rolling(window).mean()` and `returns.rolling(window).std()` work like `.mean()` and `.std()`, but over a moving window.

In [ ]:
def rolling_sharpe(returns, window=63):
    """Sharpe ratio over the last `window` days, recalculated every day."""
    return ...  # ✏️


def rolling_volatility(returns, window=30):
    """Annual volatility over the last `window` days, recalculated every day."""
    return ...  # ✏️


check(rolling_sharpe)
check(rolling_volatility)

Time to see your work. This cell draws the dashboard's four charts with your own functions: equity curve, drawdown, rolling Sharpe and rolling volatility. If one of the checks above shows ❌, fix it first, or this cell will fail too.

In [ ]:
returns = to_returns(prices)
equity = equity_curve(returns)

fig, axes = plt.subplots(2, 2, figsize=(12, 7))
equity.plot(ax=axes[0, 0], title=f"{TICKER}: growth of $1")
drawdown(equity).plot(ax=axes[0, 1], title="Drawdown", color="tab:red")
rolling_sharpe(returns).plot(ax=axes[1, 0], title="Rolling Sharpe (63 days)")
rolling_volatility(returns).plot(ax=axes[1, 1], title="Rolling volatility (30 days)")
for ax in axes.flat:
    ax.grid(alpha=0.3)
    ax.set_xlabel("")
plt.tight_layout()
plt.show()

## Step 7. CAPM: how much of it is just the market?

When the market rises, most stocks rise with it. **CAPM** (the Capital Asset Pricing Model) fits a straight line through one dot per day: the benchmark's return across, your investment's return up.
- **Beta** is the slope: 1.2 means the investment tends to move 1.2% when the market moves 1%.
- **Alpha** is the line's height where the market return is 0, times 252: the yearly return left over once the market part is taken out.
- **R squared** is how well the line fits: the share of the daily movement that the market explains.

Write `capm` so it returns these three numbers in a dict.

💡 **Hint:** first line the two series up by date with `pd.concat([returns, benchmark_returns], axis=1).dropna()`. Then beta = cov(asset, benchmark) / var(benchmark), alpha = (mean(asset) − beta × mean(benchmark)) × 252, and R squared = corr(asset, benchmark)². pandas Series have `.cov()`, `.var()` and `.corr()`.

In [ ]:
def capm(returns, benchmark_returns):
    """Alpha (annualised), beta and R squared of returns against a benchmark."""
    both = ...       # ✏️ one column each, only the dates both have
    beta = ...       # ✏️
    alpha = ...      # ✏️
    r_squared = ...  # ✏️
    return {"Alpha": alpha, "Beta": beta, "R squared": r_squared}


check(capm)

This picture is what CAPM measures. Each dot is one day: SPY's return across, your investment's return up. The red line is drawn from your alpha and beta: the steeper it is, the more the investment amplifies the market's moves.

In [ ]:
benchmark_returns = to_returns(benchmark)
result = capm(returns, benchmark_returns)
both = pd.concat([returns, benchmark_returns], axis=1, keys=["asset", "bench"]).dropna()

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(both["bench"], both["asset"], s=5, alpha=0.3)
x = np.linspace(both["bench"].min(), both["bench"].max(), 100)
ax.plot(x, result["Alpha"] / 252 + result["Beta"] * x, color="tab:red", label=f"beta = {result['Beta']:.2f}")
ax.set_xlabel("SPY daily return")
ax.set_ylabel(f"{TICKER} daily return")
ax.legend()
ax.grid(alpha=0.3)
plt.show()
pd.Series(result).round(3)

## Step 8. Put it together

The dashboard's six headline numbers come from one function. Write `tearsheet` so it returns a dict with exactly these names: `"Annual return"`, `"Annual volatility"`, `"Sharpe ratio"`, `"Sortino ratio"`, `"Max drawdown"` and `"Hit rate"`.

💡 **Hint:** call the functions you already wrote. Max drawdown is the `.min()` of your drawdown.

In [ ]:
def tearsheet(prices):
    """The dashboard's six headline numbers for one price series."""
    returns = to_returns(prices)
    return {
        "Annual return": ...,  # ✏️
        "Annual volatility": ...,
        "Sharpe ratio": ...,
        "Sortino ratio": ...,
        "Max drawdown": ...,
        "Hit rate": ...,
    }


check(tearsheet)

A good habit: check your numbers against a tool you already trust. This cell runs the workshop's backtester, whose buy & hold column measures exactly what your tearsheet does, and puts its numbers next to yours. The four numbers both tools calculate should match.

In [ ]:
from backtester import run_backtest

ticker_prices = load_prices(TICKER)
no_trades = pd.Series(0, index=ticker_prices.index)  # we only need the buy & hold column
buy_and_hold = run_backtest(ticker_prices, no_trades).benchmark_metrics
mine = tearsheet(prices)

pd.DataFrame(
    {
        "Yours": [mine["Annual return"], mine["Annual volatility"], mine["Sharpe ratio"], mine["Max drawdown"]],
        "Backtester": [buy_and_hold["Annual return (CAGR)"], buy_and_hold["Annual volatility"],
                       buy_and_hold["Sharpe ratio"], buy_and_hold["Max drawdown"]],
    },
    index=["Annual return", "Annual volatility", "Sharpe ratio", "Max drawdown"],
).round(4)

## Part 2. Turn it into a web app

Your functions work; now give them a proper home. The repo has a starter app in [`projects/trading_dashboard/`](https://github.com/kolyamkl/qm-fintech-quant-workshop/blob/main/projects/trading_dashboard): `analytics.py` is where your functions go, and `app.py` is the page, with five ✏️ TODOs. You'll edit both in a code editor and run the app from a terminal on your own computer.

**1. Get the code.** You need **Python 3.11 or newer** ([python.org/downloads](https://www.python.org/downloads/)) and **git**. Open a terminal and run:

```bash
git clone https://github.com/kolyamkl/qm-fintech-quant-workshop.git
cd qm-fintech-quant-workshop
python -m venv .venv
source .venv/bin/activate          # Windows (PowerShell): .venv\Scripts\Activate.ps1
pip install -r requirements.txt
```

(On macOS you may need `python3` instead of `python`.)

**2. Add your functions.** Open `projects/trading_dashboard/analytics.py` in any code editor (VS Code is a good free one). Replace each `raise NotImplementedError(...)` line with the body of your function from this notebook, and save.

**3. Run the app.** In the same terminal, from the repo folder:

```bash
streamlit run projects/trading_dashboard/app.py
```

Your browser opens the dashboard at `http://localhost:8501`. Stop it any time with **Ctrl + C** in the terminal.

**4. Fill in the TODOs** in `app.py`. Each TODO replaces one blue `TODO` box on the page. Save the file and refresh the browser page to see your change. If the page looks stale, stop the app with **Ctrl + C** and run the command again.

**Streamlit pieces you'll need:** `st.columns(3)`, `st.metric(label, value)`, `st.subheader(text)`, `st.line_chart(series)` and `st.area_chart(series)`. Everything else is in the [Streamlit docs](https://docs.streamlit.io/develop/api-reference). Stuck? The finished page is [`solution/app.py`](https://github.com/kolyamkl/qm-fintech-quant-workshop/blob/main/projects/trading_dashboard/solution/app.py).

## Ideas to take it further

- Let people **upload their own price CSV** with `st.file_uploader`, instead of picking a ticker.
- Add a **date range** with `st.date_input`, like the workshop app.
- Analyse a **portfolio**: two or more tickers with weights, e.g. 60% SPY and 40% TLT.
- Add a **risk-free rate** input and subtract it in Sharpe, Sortino and CAPM.
- Make the charts **interactive** with Plotly (`st.plotly_chart`). Add `plotly` to `requirements.txt` first.
- Give it **your own look** with a [Streamlit theme](https://docs.streamlit.io/develop/concepts/configuration/theming).
- **Put it online** for free: push your code to your own GitHub repo and deploy it on [Streamlit Community Cloud](https://docs.streamlit.io/deploy/streamlit-community-cloud/deploy-your-app).

Finished? Put the link on your CV and your GitHub profile: a working app you built yourself is a great thing to show.